# CommFor unseen-family evaluation: CLIP + NPR-ResNet18 + AIDE

Notebook này chạy **inference only** cho ba checkpoint trên cùng một cohort CommFor cố định: 100 ảnh real và 900 ảnh fake từ 9 generator chưa xuất hiện trong Combined Tiny-GenImage.

Khuyến nghị: chọn GPU accelerator, bật Internet cho lần cài `open_clip_torch`, và Add Input chứa runner, ba checkpoint, manifest cùng thư mục `image_cache`.

## Kaggle Input cần có

Tạo một Kaggle Dataset giữ nguyên cấu trúc sau:

```text
test_three_models_commfor_unseen_kaggle.py
checkpoints/clip_linear_head.pt
checkpoints/npr_resnet18_from_scratch.pt
checkpoints/aide/model.pt
commfor_cache/dataset/commfor_unseen_manifest.csv
commfor_cache/dataset/image_cache/000000.png ... 000999.png
```

Có thể upload thêm `image_cache_manifest.csv` và `image_cache_complete.json` để lưu provenance. Notebook tự tìm file theo tên nên tên Kaggle Dataset không bắt buộc.

In [ ]:
%pip install -q open_clip_torch datasets

In [ ]:
import os
import sys
import json
import importlib.util
from pathlib import Path

import pandas as pd
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

input_root = Path('/kaggle/input')
print('Kaggle inputs:')
for path in sorted(input_root.iterdir()):
    print(' -', path)
assert torch.cuda.is_available(), 'Hãy bật GPU accelerator trước khi chạy full.'

## Cấu hình

Giữ các đường dẫn checkpoint là `None` để notebook tự tìm theo tên file trong `/kaggle/input`. Chỉ điền đường dẫn khi Kaggle Input có nhiều file trùng tên.

Nếu đã upload cache 1.000 PNG, giữ `allow_hf_fallback=False` để notebook báo lỗi ngay khi cache thiếu, thay vì vô tình chọn lại dataset.

In [ ]:
CONFIG = {
    'input_root': '/kaggle/input',
    'output_root': '/kaggle/working/commfor_unseen_three_models',

    # None = tự tìm trong /kaggle/input
    'clip_checkpoint': None,
    'npr_checkpoint': None,
    'aide_checkpoint': None,
    'manifest_path': None,

    # CLIP checkpoint chỉ chứa linear head. None sẽ tải ViT-B-32/OpenAI.
    # Nếu Kaggle Internet OFF, upload backbone và điền path tại đây.
    'clip_backbone_path': None,

    # Đặt True chỉ khi chưa upload manifest + image_cache và muốn tải CommFor từ HF.
    'allow_hf_fallback': False,

    'fake_per_generator': 100,
    'real_per_source': 20,
    'clip_batch_size': 32,
    'npr_batch_size': 32,
}
CONFIG

### Tùy chọn: HF token

Cell này chỉ cần khi dùng fallback tải CommFor hoặc muốn tránh rate limit. Tạo Kaggle Secret tên `HF_TOKEN`; nếu không có secret thì cell tự bỏ qua.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN configured.')
except Exception:
    print('HF_TOKEN secret not configured; continuing without it.')

In [ ]:
runner_candidates = sorted(
    Path('/kaggle/input').rglob('test_three_models_commfor_unseen_kaggle.py')
)
if len(runner_candidates) != 1:
    raise FileNotFoundError(
        'Cần đúng một file test_three_models_commfor_unseen_kaggle.py trong Kaggle Input; '
        f'hiện tìm thấy {len(runner_candidates)}: {runner_candidates}'
    )
runner_path = runner_candidates[0]
print('Runner:', runner_path)
spec = importlib.util.spec_from_file_location('commfor_three_models_kaggle', runner_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
print('Runner imported successfully.')

In [ ]:
summary = runner.run_evaluation(CONFIG)
print(json.dumps(summary, ensure_ascii=False, indent=2))

## Kết quả chi tiết từng generator

In [ ]:
output_root = Path(CONFIG['output_root'])
generator_metrics = pd.read_csv(output_root / 'metrics/all_models_generator_metrics.csv')
display(
    generator_metrics[[
        'model', 'generator', 'generator_family', 'num_real', 'num_fake',
        'balanced_accuracy', 'real_recall', 'fake_recall',
        'fake_precision', 'fake_f1', 'roc_auc', 'average_precision'
    ]].sort_values(['generator', 'model'])
)

In [ ]:
summary_table = pd.read_csv(output_root / 'metrics/all_models_summary.csv')
display(summary_table)

## Đóng gói output

File ZIP xuất hiện trong tab Output của Kaggle sau khi cell hoàn tất.

In [ ]:
import shutil
zip_path = shutil.make_archive(
    '/kaggle/working/commfor_unseen_three_models_results',
    'zip',
    root_dir=output_root,
)
print('Download:', zip_path)